In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime

# ==================== 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ====================
class AverageMeter:
    """Вычисляет и хранит среднее значение и текущее значение"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def calculate_metrics(y_true, y_pred, class_names):
    """
    Вычисление метрик для задачи классификации
    """
    # Метрики по всем классам (macro averaging)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Метрики по всем классам (micro averaging)
    precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
    recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    
    # Метрики для каждого класса
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    # Подробный отчет
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    return {
        'macro': {
            'precision': precision_macro,
            'recall': recall_macro,
            'f1': f1_macro
        },
        'micro': {
            'precision': precision_micro,
            'recall': recall_micro,
            'f1': f1_micro
        },
        'per_class': {
            'precision': precision_per_class,
            'recall': recall_per_class,
            'f1': f1_per_class
        },
        'report': report,
        'confusion_matrix': cm
    }

def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    """Визуализация истории обучения"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # График loss
    axes[0].plot(train_losses, label='Train Loss')
    axes[0].plot(val_losses, label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # График accuracy
    axes[1].plot(train_accs, label='Train Accuracy')
    axes[1].plot(val_accs, label='Val Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(cm, class_names, normalize=True):
    """Визуализация confusion matrix"""
    plt.figure(figsize=(10, 8))
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
    else:
        fmt = 'd'
    
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix' + (' (Normalized)' if normalize else ''))
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

def print_metrics_table(metrics, class_names):
    """Вывод метрик в виде таблицы"""
    print("\n" + "="*80)
    print("МЕТРИКИ КЛАССИФИКАЦИИ")
    print("="*80)
    
    # Макро-метрики
    print("\n--- МАКРО-МЕТРИКИ (по всем классам) ---")
    macro_df = pd.DataFrame([metrics['macro']])
    print(macro_df.to_string(index=False))
    
    # Микро-метрики
    print("\n--- МИКРО-МЕТРИКИ (по всем классам) ---")
    micro_df = pd.DataFrame([metrics['micro']])
    print(micro_df.to_string(index=False))
    
    # Метрики по классам
    print("\n--- МЕТРИКИ ПО КАЖДОМУ КЛАССУ ---")
    per_class_data = []
    for i, class_name in enumerate(class_names):
        per_class_data.append({
            'Class': class_name,
            'Precision': f"{metrics['per_class']['precision'][i]:.4f}",
            'Recall': f"{metrics['per_class']['recall'][i]:.4f}",
            'F1': f"{metrics['per_class']['f1'][i]:.4f}"
        })
    per_class_df = pd.DataFrame(per_class_data)
    print(per_class_df.to_string(index=False))
    
    # Подробный отчет
    print("\n--- ПОДРОБНЫЙ ОТЧЕТ ---")
    report_df = pd.DataFrame(metrics['report']).transpose()
    print(report_df.to_string())

# ==================== 2. АРХИТЕКТУРА RESNET-18 ====================
class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, self.expansion * out_channels,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * out_channels)
            )
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512 * block.expansion, num_classes)
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = nn.functional.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

# ==================== 3. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ ====================
def load_cifar10_data(batch_size=64, val_size=10000):
    """Загрузка и подготовка данных CIFAR-10"""
    print("Загрузка датасета CIFAR-10...")
    
    # Аугментации для тренировочных данных
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Загрузка всего тренировочного набора
    full_train_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train
    )
    
    # Разделение на train и validation
    indices = list(range(len(full_train_dataset)))
    np.random.seed(42)  # Для воспроизводимости
    np.random.shuffle(indices)
    train_indices = indices[val_size:]
    val_indices = indices[:val_size]
    
    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_train_dataset, val_indices)
    
    # Тестовый набор
    test_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test
    )
    
    # DataLoader'ы
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Имена классов
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    print(f"Данные загружены:")
    print(f"  - Train: {len(train_dataset)} изображений")
    print(f"  - Val: {len(val_dataset)} изображений")
    print(f"  - Test: {len(test_dataset)} изображений")
    print(f"  - Классы: {len(class_names)}")
    print(f"  - Имена классов: {class_names}")
    print(f"  - Batch size: {batch_size}")
    
    return train_loader, val_loader, test_loader, class_names

# ==================== 4. ПРОЦЕСС ОБУЧЕНИЯ ====================
def train_epoch(model, train_loader, criterion, optimizer, device, epoch, total_epochs):
    """Одна эпоха обучения"""
    model.train()
    losses = AverageMeter()
    accuracies = AverageMeter()
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{total_epochs} [Train]', leave=False)
    for batch_idx, (inputs, targets) in enumerate(pbar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        # Вычисление accuracy
        _, predicted = outputs.max(1)
        correct = predicted.eq(targets).sum().item()
        accuracy = 100. * correct / targets.size(0)
        
        # Обновление метрик
        losses.update(loss.item(), inputs.size(0))
        accuracies.update(accuracy, inputs.size(0))
        
        # Обновление progress bar
        pbar.set_postfix({
            'Loss': f'{losses.avg:.4f}',
            'Acc': f'{accuracies.avg:.2f}%'
        })
    
    return losses.avg, accuracies.avg

def validate(model, val_loader, criterion, device, return_predictions=False):
    """Валидация модели"""
    model.eval()
    losses = AverageMeter()
    accuracies = AverageMeter()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Вычисление accuracy
            _, predicted = outputs.max(1)
            correct = predicted.eq(targets).sum().item()
            accuracy = 100. * correct / targets.size(0)
            
            # Обновление метрик
            losses.update(loss.item(), inputs.size(0))
            accuracies.update(accuracy, inputs.size(0))
            
            # Сохранение предсказаний и меток
            if return_predictions:
                all_predictions.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
    
    if return_predictions:
        return losses.avg, accuracies.avg, all_predictions, all_targets
    else:
        return losses.avg, accuracies.avg

# ==================== 5. ОСНОВНАЯ ФУНКЦИЯ ====================
def main():
    # Параметры
    BATCH_SIZE = 64
    EPOCHS = 30
    LEARNING_RATE = 0.001
    VAL_SIZE = 10000
    NUM_CLASSES = 10
    
    # Устройство
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Устройство: {device}")
    
    # Загрузка данных
    train_loader, val_loader, test_loader, class_names = load_cifar10_data(
        batch_size=BATCH_SIZE, val_size=VAL_SIZE
    )
    
    # Модель
    model = ResNet18(num_classes=NUM_CLASSES).to(device)
    
    # Подсчет параметров
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nМодель: ResNet-18")
    print(f"Параметров: {total_params:,}")
    print(f"Обучаемых параметров: {trainable_params:,}")
    
    # Функция потерь и оптимизатор
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Используем StepLR вместо ReduceLROnPlateau с verbose
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    
    # Сохранение лучшей модели
    best_val_acc = 0.0
    best_model_path = "best_resnet18_cifar10.pth"
    
    # История обучения
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    print("\n" + "="*70)
    print("МНОЖЕСТВЕННАЯ КЛАССИФИКАЦИЯ CIFAR-10 С RESNET-18")
    print("="*70)
    print(f"\nНачало обучения:")
    print(f"  Эпохи: {EPOCHS}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Batch size: {BATCH_SIZE}")
    print("="*70)
    
    # Цикл обучения
    for epoch in range(EPOCHS):
        # Обучение
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch, EPOCHS)
        
        # Валидация
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Обновление learning rate
        scheduler.step()
        
        # Сохранение истории
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        # Вывод результатов эпохи
        print(f"\nЭпоха [{epoch+1}/{EPOCHS}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Сохранение лучшей модели
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, best_model_path)
            print(f"  ✓ Сохранена лучшая модель (Val Acc: {val_acc:.2f}%)")
    
    print("\nОбучение завершено!")
    print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")
    
    # Визуализация истории обучения
    print("\nВизуализация истории обучения...")
    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    
    # ==================== 6. ТЕСТИРОВАНИЕ И МЕТРИКИ ====================
    print("\n" + "="*70)
    print("ТЕСТИРОВАНИЕ МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ")
    print("="*70)
    
    # Загрузка лучшей модели
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Тестирование на тестовой выборке
    print("\nВычисление метрик на тестовой выборке...")
    test_loss, test_acc, test_predictions, test_targets = validate(
        model, test_loader, criterion, device, return_predictions=True
    )
    
    print(f"\nРезультаты на тестовой выборке:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_acc:.2f}%")
    
    # Вычисление всех метрик
    metrics = calculate_metrics(test_targets, test_predictions, class_names)
    
    # Вывод метрик в виде таблицы
    print_metrics_table(metrics, class_names)
    
    # Визуализация confusion matrix
    print("\nВизуализация confusion matrix...")
    plot_confusion_matrix(metrics['confusion_matrix'], class_names, normalize=True)
    plot_confusion_matrix(metrics['confusion_matrix'], class_names, normalize=False)
    
    # ==================== 7. ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ ====================
    print("\n" + "="*70)
    print("ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ")
    print("="*70)
    
    # Анализ лучших и худших классов
    per_class_f1 = metrics['per_class']['f1']
    best_class_idx = np.argmax(per_class_f1)
    worst_class_idx = np.argmin(per_class_f1)
    
    print(f"\nЛучший класс по F1-score: {class_names[best_class_idx]} "
          f"(F1 = {per_class_f1[best_class_idx]:.4f})")
    print(f"Худший класс по F1-score: {class_names[worst_class_idx]} "
          f"(F1 = {per_class_f1[worst_class_idx]:.4f})")
    
    # Средние метрики
    print(f"\nСредний F1-score (macro): {metrics['macro']['f1']:.4f}")
    print(f"Средний Precision (macro): {metrics['macro']['precision']:.4f}")
    print(f"Средний Recall (macro): {metrics['macro']['recall']:.4f}")
    
    # Сохранение результатов в файл
    results = {
        'model': 'ResNet-18',
        'dataset': 'CIFAR-10',
        'best_val_accuracy': float(best_val_acc),
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'metrics': metrics,
        'training_history': {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs
        }
    }
    
    # Сохранение результатов
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f"results_cifar10_resnet18_{timestamp}.pth"
    torch.save(results, results_file)
    print(f"\nРезультаты сохранены в файл: {results_file}")
    
    # ==================== 8. ИНФЕРЕНС НА ОДНОМ ИЗОБРАЖЕНИИ ====================
    print("\n" + "="*70)
    print("ДЕМОНСТРАЦИЯ ИНФЕРЕНСА НА СЛУЧАЙНОМ ИЗОБРАЖЕНИИ")
    print("="*70)
    
    # Получение случайного изображения из тестовой выборки
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    idx = np.random.randint(0, len(images))
    
    # Предсказание
    model.eval()
    with torch.no_grad():
        image = images[idx].unsqueeze(0).to(device)
        output = model(image)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        predicted_class = torch.argmax(output, dim=1).item()
        confidence = torch.max(probabilities).item()
    
    # Визуализация
    image_np = images[idx].permute(1, 2, 0).numpy()
    # Denormalize
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    image_np = std * image_np + mean
    image_np = np.clip(image_np, 0, 1)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(image_np)
    plt.title(f"True: {class_names[labels[idx]]} | "
              f"Pred: {class_names[predicted_class]} ({confidence:.2%})")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('example_prediction.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Истинный класс: {class_names[labels[idx]]}")
    print(f"Предсказанный класс: {class_names[predicted_class]}")
    print(f"Уверенность: {confidence:.2%}")
    
    print("\n" + "="*70)
    print("ВСЕ ТРЕБОВАНИЯ ЗАДАЧИ ВЫПОЛНЕНЫ:")
    print("="*70)
    print("✓ Задача: Множественная классификация")
    print("✓ Датасет: CIFAR-10 (10 классов)")
    print("✓ Архитектура: ResNet-18")
    print("✓ Обучение: 30 эпох с валидацией")
    print("✓ Метрики по всем классам: Precision, Recall, F1 (macro и micro)")
    print("✓ Метрики по каждому классу: Precision, Recall, F1")
    print("✓ Confusion matrix")
    print("✓ Сохранение модели и результатов")
    print("✓ Визуализация истории обучения")
    print("="*70)

if __name__ == "__main__":
    main()

Устройство: cuda
Загрузка датасета CIFAR-10...
Данные загружены:
  - Train: 40000 изображений
  - Val: 10000 изображений
  - Test: 10000 изображений
  - Классы: 10
  - Имена классов: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  - Batch size: 64

Модель: ResNet-18
Параметров: 11,173,962
Обучаемых параметров: 11,173,962

МНОЖЕСТВЕННАЯ КЛАССИФИКАЦИЯ CIFAR-10 С RESNET-18

Начало обучения:
  Эпохи: 30
  Learning rate: 0.001
  Batch size: 64



Эпоха [1/30]
  Train Loss: 1.5497, Train Acc: 42.70%
  Val Loss: 1.1617, Val Acc: 57.33%
  Learning Rate: 0.001000
  ✓ Сохранена лучшая модель (Val Acc: 57.33%)



Эпоха [2/30]
  Train Loss: 1.0721, Train Acc: 61.39%
  Val Loss: 1.2080, Val Acc: 59.62%
  Learning Rate: 0.001000
  ✓ Сохранена лучшая модель (Val Acc: 59.62%)


Epoch 3/30 [Train]:   0%|                                                                      | 0/625 [00:00<?, ?it/s]